# RF-DETR Multi-Class Detection Pipeline: Auto-Discovered Categories
### Model: RF-DETR Base (`resolution=560`, `optimizer=adam`, `lr=5e-5`, `lr_scheduler=cosine`)
This pipeline fine-tunes an RF-DETR Base model on multi-class annotations with:
- **Dual Output Logging**: Informative, user-friendly `print()` progress statements + persistent `pipeline.log` records
- **Category-Stratified Splits**: Equal category proportions maintained strictly across train, validation, and test sets
- **Sample vs Full Mode**: Toggle `SAMPLE_SIZE = 1000` for fast pipeline testing vs `SAMPLE_SIZE = None` for complete dataset training
- **Early Visual EDA**: Render sample images with bounding boxes & distinct multi-class colors using `supervision`
- **Dynamic Folders**: Creates split folders, logs, checkpoints, and inference outputs inside this pipeline folder
- **Post-Training Optimization**: `optimize_for_inference()` and automated best checkpoint copying

In [ ]:
# STEP 0 — Install Dependencies (RF-DETR 1.4.0 & Stack)
print("=" * 70)
print("📦 [STEP 0] Verifying and installing dependencies...")
print("=" * 70)

!pip install -q --no-cache-dir "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu121
!pip install -q --no-cache-dir "rfdetr==1.4.0"
!pip install -q --no-cache-dir "supervision>=0.22.0"
!pip install -q --no-cache-dir pycocotools pandas numpy opencv-python Pillow matplotlib python-dotenv azure-storage-blob requests tqdm

print("✅ Dependencies verified successfully.")


In [ ]:
# CELL 1 — Imports & Dual Logger Initialization
import os
import sys
import json
import logging
import random
import shutil
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import requests
import torch
from dotenv import load_dotenv
import supervision as sv

try:
    from rfdetr import RFDETRBase
    rfdetr_status = "Available (rfdetr.RFDETRBase)"
except ImportError:
    rfdetr_status = "Not installed yet - run Step 0"

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()

log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

print("=" * 70)
print("🚀 [CELL 1] Core Libraries & Logger Initialized")
print("=" * 70)
print(f"   • Python Version:      {sys.version.split()[0]}")
print(f"   • PyTorch Version:     {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"   • Supervision Version: {sv.__version__}")
print(f"   • RF-DETR Module:      {rfdetr_status}")
logger.info("Logger initialized successfully.")


In [ ]:
# CELL 2 — Configuration & Mode Setup
# ==============================================================================
# SAMPLE vs FULL DATA MODE:
# Set SAMPLE_SIZE = 1000 to sample 1,000 images for fast testing and flow verification.
# Set SAMPLE_SIZE = None to run on the complete full dataset.
# ==============================================================================
SAMPLE_SIZE = 1000  # Set to None for full dataset training

PIPELINE_NAME = "multi_class_train_rfdetr"
MODE_TAG = f"sample_{SAMPLE_SIZE}" if SAMPLE_SIZE else "full_data"

# Working directories
REPO_ROOT = Path(".")
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = REPO_ROOT / "coco_files"
DATASET_DIR = PIPELINE_DIR / f"dataset_{MODE_TAG}"
OUTPUT_DIR = PIPELINE_DIR / f"output_{MODE_TAG}"
INFERENCE_OUTPUT_DIR = PIPELINE_DIR / f"inference_{MODE_TAG}"
MODEL_DIR = PIPELINE_DIR / "model"

PRETRAINED_WEIGHTS = "/home/jupyter/rf-detr-base-coco.pth"
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"

IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"
BBOX_FORMAT = "xywh"

# Model Hyperparameters
RESOLUTION = 560
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPTIMIZER = "adam"
LR_SCHEDULER = "cosine"
EPOCHS = 50
BATCH_SIZE = 2             # Safe batch size to prevent CUDA OOM on T4 GPUs
LR = 5e-5                  # Smaller learning rate for Adam fine-tuning
WEIGHT_DECAY = 1e-4        # Adam weight decay regularization
NUM_WORKERS = 2
GRAD_ACCUM_STEPS = 1

# Early Stopping
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.001

TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42

DOWNLOAD_WORKERS = 16
CONFIDENCE = 0.50
NMS_THRESHOLD = 0.50
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

print("=" * 70)
print(f"⚙️  [CELL 2] Configuration Loaded: [{PIPELINE_NAME}]")
print("=" * 70)
print(f"   • Execution Mode:       {MODE_TAG.upper()} ({SAMPLE_SIZE if SAMPLE_SIZE else 'Full Data'} images)")
print(f"   • Multi-Class Strategy: Auto-Discovery with Category Stratification")
print(f"   • Input Resolution:     {RESOLUTION}x{RESOLUTION}")
print(f"   • Optimizer & Schedule: {OPTIMIZER.upper()} (lr={LR}, weight_decay={WEIGHT_DECAY}), Cosine Decay")
print(f"   • Early Stopping:       Patience={EARLY_STOPPING_PATIENCE}, Min Delta={EARLY_STOPPING_MIN_DELTA}")
print(f"   • Post-processing:      Confidence={CONFIDENCE}, NMS IoU={NMS_THRESHOLD}")
print(f"   • Output Folder:        {PIPELINE_DIR}")
logger.info(f"Configuration loaded for {PIPELINE_NAME} ({MODE_TAG}).")


In [ ]:
# CELL 3 — Dynamic Folder Creation & File Logging
print("=" * 70)
print("📁 [CELL 3] Creating pipeline directories dynamically...")
print("=" * 70)

for p in [DATASET_DIR, OUTPUT_DIR, INFERENCE_OUTPUT_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    print(f"   📁 Ready: {p}")

log_file_path = OUTPUT_DIR / "pipeline.log"
file_handler = logging.FileHandler(log_file_path, mode="a", encoding="utf-8")
file_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(file_handler)

load_dotenv()
connection_string = os.getenv(AZURE_CONNECTION_STRING_ENV)
try:
    from azure.storage.blob import BlobServiceClient
    blob_service_client = BlobServiceClient.from_connection_string(connection_string) if connection_string else None
    if blob_service_client:
        print("   ☁️  Azure BlobServiceClient initialized.")
    else:
        print("   ⚠️  Azure connection string not set. Will use local image files if available.")
except Exception as e:
    blob_service_client = None
    print(f"   ℹ️  Azure SDK note: {e}")

print(f"✅ Dynamic directories prepared. File logger writing to {log_file_path}")
logger.info(f"Dynamic directories created for {PIPELINE_NAME}.")


In [ ]:
# CELL 4 — Read all annotation JSON files from coco_files/
print("=" * 70)
print(f"📄 [CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
print("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
print(f"   • Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    print(f"   • Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

print(f"✅ Total raw annotation records loaded: {len(raw_records)}")
logger.info(f"Loaded {len(raw_records)} raw annotation records from {len(json_files)} file(s).")


In [ ]:
# CELL 5 — Normalize Annotations & Auto-Discover Multiple Categories
print("=" * 70)
print("🔍 [CELL 5] Parsing annotations and discovering categories dynamically...")
print("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

print(f"   • Unique images found:         {len(image_annotations)}")
print(f"   • Total valid bounding boxes:  {total_boxes}")
print(f"   • Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    print(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")


In [ ]:
# CELL 6 — Group Annotations & Apply Stratified Sampling Mode
print("=" * 70)
print(f"📊 [CELL 6] Applying mode selection: {MODE_TAG.upper()}")
print("=" * 70)

all_image_urls = sorted(list(image_annotations.keys()))
random.seed(RANDOM_SEED)

# Map each image to its primary category for stratification
image_primary_cat: Dict[str, str] = {}
for u in all_image_urls:
    anns = image_annotations[u]
    if anns:
        cats = [a["category_name"] for a in anns]
        image_primary_cat[u] = max(set(cats), key=cats.count)
    else:
        image_primary_cat[u] = AUTO_CATEGORIES[0]

if SAMPLE_SIZE is not None and len(all_image_urls) > SAMPLE_SIZE:
    selected_image_urls = []
    cat_to_images: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
    for u in all_image_urls:
        cat_to_images[image_primary_cat[u]].append(u)
    
    total_imgs = len(all_image_urls)
    for cat in AUTO_CATEGORIES:
        imgs = cat_to_images[cat]
        target_count = max(1, int(round((len(imgs) / total_imgs) * SAMPLE_SIZE)))
        selected = random.sample(imgs, min(len(imgs), target_count))
        selected_image_urls.extend(selected)
    
    if len(selected_image_urls) > SAMPLE_SIZE:
        selected_image_urls = random.sample(selected_image_urls, SAMPLE_SIZE)
    print(f"   🎯 Sample Mode Active: Stratified selection of {len(selected_image_urls)} images.")
else:
    selected_image_urls = all_image_urls
    print(f"   🌐 Full Data Mode: Using all {len(selected_image_urls)} images.")

image_urls = sorted(list(set(selected_image_urls)))
filtered_annotations = {u: image_annotations[u] for u in image_urls}
total_selected_boxes = sum(len(v) for v in filtered_annotations.values())

print(f"   • Total images in pipeline:        {len(image_urls)}")
print(f"   • Total bounding boxes in pipeline: {total_selected_boxes}")
logger.info(f"Mode [{MODE_TAG}]: {len(image_urls)} images, {total_selected_boxes} annotations.")


In [ ]:
# CELL 7 — Azure URL Helpers & Download Function
print("=" * 70)
print("☁️  [CELL 7] Defining Azure Blob download and caching functions...")
print("=" * 70)

RAW_IMAGES_DIR = PIPELINE_DIR / "raw_images"
RAW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def extract_blob_info(url: str) -> Tuple[Optional[str], Optional[str], str]:
    parsed = urlparse(url)
    clean_path = parsed.path.lstrip("/")
    parts = clean_path.split("/", 1)
    filename = Path(clean_path).name.split("?")[0]
    if len(parts) == 2:
        return parts[0], parts[1], filename
    return None, None, filename

def download_image(url: str, output_dir: Path) -> Tuple[str, Optional[Path], Optional[str]]:
    container, blob_name, filename = extract_blob_info(url)
    dest_path = output_dir / filename
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return url, dest_path, None
    if blob_service_client and container and blob_name:
        try:
            bc = blob_service_client.get_blob_client(container=container, blob=blob_name)
            with open(dest_path, "wb") as f:
                f.write(bc.download_blob().readall())
            return url, dest_path, None
        except Exception as e:
            pass
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with open(dest_path, "wb") as f:
            f.write(resp.content)
        return url, dest_path, None
    except Exception as e:
        return url, None, str(e)

print("✅ Image download functions ready.")


In [ ]:
# CELL 8 — Parallel Azure Image Downloads
print("=" * 70)
print(f"⬇️  [CELL 8] Downloading {len(image_urls)} images ({DOWNLOAD_WORKERS} threads)...")
print("=" * 70)

download_results: Dict[str, Dict[str, Any]] = {}
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {executor.submit(download_image, url, RAW_IMAGES_DIR): url for url in image_urls}
    done_count = 0
    for future in as_completed(futures):
        url, path, err = future.result()
        download_results[url] = {"local_path": path, "error": err}
        done_count += 1
        if done_count % 200 == 0 or done_count == len(image_urls):
            print(f"   • Progress: {done_count}/{len(image_urls)} processed...")

success_downloads = [u for u, res in download_results.items() if res["error"] is None]
failed_downloads = [u for u, res in download_results.items() if res["error"] is not None]

print(f"✅ Download Summary:")
print(f"   • Successfully ready: {len(success_downloads)} images")
print(f"   • Failed:             {len(failed_downloads)} images")
logger.info(f"Downloaded {len(success_downloads)} images, failed: {len(failed_downloads)}.")


In [ ]:
# CELL 9 — Early Visual EDA: Print Images with BBoxes & Multiple Categories in the Beginning
print("=" * 70)
print("🎨 [CELL 9] Early Visual EDA: Displaying sample images with multi-class bounding boxes...")
print("=" * 70)

valid_downloaded_urls = [u for u in image_urls if u in download_results and download_results[u]["error"] is None]
eda_color_palette = sv.ColorPalette.from_hex([
    "#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"
])

def display_eda_samples(sample_urls, max_samples=3):
    rendered = 0
    for url in sample_urls[:max_samples]:
        img_path = download_results[url]["local_path"]
        if not (img_path and img_path.exists()):
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        anns = filtered_annotations.get(url, [])
        if not anns:
            continue

        xyxy_boxes = []
        class_ids = []
        labels = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            xyxy_boxes.append([x, y, x + w, y + h])
            cid = ann["category_id"]
            cname = ann["category_name"]
            class_ids.append(cid)
            labels.append(cname)

        detections = sv.Detections(
            xyxy=np.array(xyxy_boxes, dtype=np.float32),
            class_id=np.array(class_ids, dtype=int)
        )
        box_annotator = sv.BoxAnnotator(color=eda_color_palette, thickness=3)
        label_annotator = sv.LabelAnnotator(color=eda_color_palette, text_scale=0.6, text_thickness=2)
        annotated = box_annotator.annotate(scene=img.copy(), detections=detections)
        annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title(f"Multi-Class Visual EDA: {img_path.name} ({len(anns)} boxes)", fontsize=12)
        plt.axis("off")
        plt.show()
        rendered += 1
    print(f"   • Rendered {rendered} visual sample(s) with multi-class supervision bounding boxes.")

if valid_downloaded_urls:
    display_eda_samples(valid_downloaded_urls, max_samples=3)
else:
    print("   ℹ️  No downloaded images available for inline rendering.")


In [ ]:
# CELL 10 — Category-Stratified Train/Val/Test Split (Equal Proportions)
print("=" * 70)
print("✂️  [CELL 10] Performing Category-Stratified Split (Train 80%, Val 10%, Test 10%)...")
print("=" * 70)

valid_urls = [u for u in image_urls if download_results.get(u, {}).get("error") is None]
random.seed(RANDOM_SEED)

cat_to_valid_imgs: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
for u in valid_urls:
    p_cat = image_primary_cat.get(u, AUTO_CATEGORIES[0])
    cat_to_valid_imgs[p_cat].append(u)

train_urls, val_urls, test_urls = [], [], []
for cat, imgs in cat_to_valid_imgs.items():
    random.shuffle(imgs)
    n = len(imgs)
    n_tr = int(n * TRAIN_RATIO)
    n_va = int(n * VALID_RATIO)
    train_urls.extend(imgs[:n_tr])
    val_urls.extend(imgs[n_tr:n_tr + n_va])
    test_urls.extend(imgs[n_tr + n_va:])

def count_category_boxes(urls: List[str]) -> Dict[str, int]:
    counts = {cat: 0 for cat in AUTO_CATEGORIES}
    for u in urls:
        for a in filtered_annotations.get(u, []):
            c = a["category_name"]
            counts[c] = counts.get(c, 0) + 1
    return counts

train_cat_boxes = count_category_boxes(train_urls)
val_cat_boxes = count_category_boxes(val_urls)
test_cat_boxes = count_category_boxes(test_urls)
total_cat_boxes = {cat: train_cat_boxes[cat] + val_cat_boxes[cat] + test_cat_boxes[cat] for cat in AUTO_CATEGORIES}

proportions_rows = []
for cat in AUTO_CATEGORIES:
    tot = max(1, total_cat_boxes[cat])
    proportions_rows.append({
        "Category": cat,
        "Total Boxes": total_cat_boxes[cat],
        "Train Count": train_cat_boxes[cat],
        "Train %": f"{(train_cat_boxes[cat]/tot)*100:.1f}%",
        "Val Count": val_cat_boxes[cat],
        "Val %": f"{(val_cat_boxes[cat]/tot)*100:.1f}%",
        "Test Count": test_cat_boxes[cat],
        "Test %": f"{(test_cat_boxes[cat]/tot)*100:.1f}%",
    })

proportions_df = pd.DataFrame(proportions_rows)
print("📊 Category Proportions Across Splits:")
print(proportions_df.to_string(index=False))
print(f"\n   • Total Images: Train={len(train_urls)}, Val={len(val_urls)}, Test={len(test_urls)}")
logger.info(f"Stratified split verified across {len(AUTO_CATEGORIES)} categories.")


In [ ]:
# CELL 11 — Create Split Folders & Copy Images
print("=" * 70)
print("📂 [CELL 11] Organizing split image directories...")
print("=" * 70)

split_dirs = {}
for split_name in ["train", "val", "test"]:
    s_dir = DATASET_DIR / split_name / "images"
    s_dir.mkdir(parents=True, exist_ok=True)
    split_dirs[split_name] = s_dir

url_to_split_map = {}
for split_name, u_list in [("train", train_urls), ("val", val_urls), ("test", test_urls)]:
    for url in u_list:
        url_to_split_map[url] = split_name
        src_path = download_results[url]["local_path"]
        if src_path and src_path.exists():
            dest = split_dirs[split_name] / src_path.name
            if not dest.exists():
                shutil.copy2(src_path, dest)

print(f"   • Train images directory: {split_dirs['train']} ({len(list(split_dirs['train'].glob('*.*')))} images)")
print(f"   • Val images directory:   {split_dirs['val']} ({len(list(split_dirs['val'].glob('*.*')))} images)")
print(f"   • Test images directory:  {split_dirs['test']} ({len(list(split_dirs['test'].glob('*.*')))} images)")
print("✅ Images organized successfully.")


In [ ]:
# CELL 12 — Build Multi-Class COCO Annotations with Auto-Discovered Categories
print("=" * 70)
print(f"📝 [CELL 12] Building COCO annotations for {NUM_CLASSES} discovered categories...")
print("=" * 70)

categories_def = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]

def build_coco_for_split(urls: List[str], split_name: str) -> Dict[str, Any]:
    images_list = []
    annotations_list = []
    ann_id = 1

    for img_id_idx, url in enumerate(urls, start=1):
        local_path = download_results[url]["local_path"]
        if not (local_path and local_path.exists()):
            continue
        try:
            with Image.open(local_path) as im:
                width, height = im.size
        except Exception:
            width, height = 1920, 1080

        filename = local_path.name
        images_list.append({
            "id": img_id_idx,
            "file_name": filename,
            "width": int(width),
            "height": int(height),
            "original_url": url
        })

        for ann in filtered_annotations.get(url, []):
            x, y, w, h = ann["bbox"]
            cid = ann["category_id"]
            annotations_list.append({
                "id": ann_id,
                "image_id": img_id_idx,
                "category_id": int(cid),
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
                "segmentation": []
            })
            ann_id += 1

    coco_dict = {
        "info": {"description": f"RF-DETR Multi-Class Dataset ({NUM_CLASSES} classes)", "version": "1.0"},
        "licenses": [],
        "images": images_list,
        "annotations": annotations_list,
        "categories": categories_def
    }
    
    out_file = DATASET_DIR / split_name / "_annotations.coco.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(coco_dict, f, indent=2)
    print(f"   • {split_name.upper():5s} COCO saved: {out_file} ({len(images_list)} images, {len(annotations_list)} boxes)")
    return coco_dict

train_coco = build_coco_for_split(train_urls, "train")
val_coco = build_coco_for_split(val_urls, "val")
test_coco = build_coco_for_split(test_urls, "test")

print("✅ Multi-Class COCO JSON files created.")


In [ ]:
# CELL 13 — Dataset Validation
print("=" * 70)
print("🔎 [CELL 13] Running COCO dataset validation checks...")
print("=" * 70)

valid_category_ids = set(range(NUM_CLASSES))
for split_name in ["train", "val", "test"]:
    ann_file = DATASET_DIR / split_name / "_annotations.coco.json"
    with open(ann_file, "r") as f:
        data = json.load(f)
    n_imgs = len(data["images"])
    n_anns = len(data["annotations"])
    cat_ids = set(a["category_id"] for a in data["annotations"])
    print(f"   • {split_name.upper():5s} Check: {n_imgs} images, {n_anns} annotations, Category IDs: {cat_ids} -> ALL VALID")
    assert cat_ids.issubset(valid_category_ids), f"Unexpected category ID in {split_name}"

print("✅ Dataset validation passed.")


In [ ]:
# CELL 14 — Verify Pretrained Weights
print("=" * 70)
print("⚖️  [CELL 14] Verifying RF-DETR Base pretrained checkpoint weights...")
print("=" * 70)

weights_path = Path(PRETRAINED_WEIGHTS)
if weights_path.exists():
    print(f"   • Pretrained weights verified at: {weights_path} ({weights_path.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"   ℹ️  Local weights {PRETRAINED_WEIGHTS} not found. RF-DETR will auto-download 'rf-detr-base-coco.pth'.")


In [ ]:
# CELL 15 — Initialize RF-DETR Base Model (Resolution 560, num_classes)
print("=" * 70)
print(f"🏗️  [CELL 15] Initializing RF-DETR Base Architecture ({NUM_CLASSES} Classes)...")
print("=" * 70)

model = RFDETRBase(
    pretrain_weights=PRETRAINED_WEIGHTS if Path(PRETRAINED_WEIGHTS).exists() else "rf-detr-base-coco.pth",
    resolution=RESOLUTION,
    device=DEVICE,
    num_classes=NUM_CLASSES
)

print(f"   • Model:        RFDETRBase")
print(f"   • Resolution:   {RESOLUTION}x{RESOLUTION}")
print(f"   • Num Classes:  {NUM_CLASSES} ({AUTO_CATEGORIES})")
print(f"   • Device:       {DEVICE}")
print("✅ RF-DETR Base model instantiated.")
logger.info(f"RF-DETR Base model initialized with {NUM_CLASSES} classes at resolution={RESOLUTION} on {DEVICE}.")


In [ ]:
# CELL 16 — Training Configuration with Adam, Smaller LR 5e-5, Cosine Decay
print("=" * 70)
print("📋 [CELL 16] Preparing training configuration and optimizer arguments...")
print("=" * 70)

train_kwargs = {
    "dataset_dir": str(DATASET_DIR),
    "train_split": "train",
    "val_split": "val",
    "annotation_file": "_annotations.coco.json",
    "image_folder": "images",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "optimizer": OPTIMIZER,
    "lr_scheduler": LR_SCHEDULER,
    "num_workers": NUM_WORKERS,
    "output_dir": str(OUTPUT_DIR),
    "early_stopping": EARLY_STOPPING,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
}

formatted_train_kwargs = json.dumps(train_kwargs, indent=2)
print(formatted_train_kwargs)
logger.info("Training Arguments:\n" + formatted_train_kwargs)


In [ ]:
# CELL 17 — Train RF-DETR Base Model
print("=" * 70)
print(f"🔥 [CELL 17] Starting RF-DETR Base Multi-Class Fine-Tuning ({NUM_CLASSES} Classes)...")
print("=" * 70)

logger.info("Starting model training...")
train_result = model.train(**train_kwargs)

print("✅ RF-DETR Training completed.")
logger.info("Training completed.")


In [ ]:
# CELL 18 — Select Best Checkpoint and Copy to model/ Directory
print("=" * 70)
print("🏆 [CELL 18] Inspecting trained checkpoints...")
print("=" * 70)

checkpoint_candidates = [
    OUTPUT_DIR / "checkpoint_best_total.pth",
    OUTPUT_DIR / "checkpoint_best_regular.pth",
    OUTPUT_DIR / "checkpoint_best_ema.pth",
    OUTPUT_DIR / "best_model.pth",
    OUTPUT_DIR / "checkpoint.pth"
]

best_checkpoint = None
for cp in checkpoint_candidates:
    if cp.exists():
        best_checkpoint = cp
        break

if not best_checkpoint:
    pth_files = sorted(list(OUTPUT_DIR.glob("*.pth")), key=lambda p: p.stat().st_mtime, reverse=True)
    if pth_files:
        best_checkpoint = pth_files[0]

dest_checkpoint = MODEL_DIR / f"best_model_{MODE_TAG}.pth"
if best_checkpoint and best_checkpoint.exists():
    shutil.copy2(best_checkpoint, dest_checkpoint)
    print(f"   • Best checkpoint identified: {best_checkpoint.name} ({best_checkpoint.stat().st_size / 1e6:.1f} MB)")
    print(f"   • Exported to:                {dest_checkpoint}")
    logger.info(f"Exported best checkpoint to {dest_checkpoint}")
else:
    print("   ⚠️  No checkpoint file found in output directory yet.")


In [ ]:
# CELL 19 — Load Trained Model & Optimize for Inference
print("=" * 70)
print("⚡ [CELL 19] Loading trained checkpoint and optimizing for inference...")
print("=" * 70)

inference_model = RFDETRBase(
    pretrain_weights=str(dest_checkpoint) if dest_checkpoint.exists() else str(best_checkpoint),
    resolution=RESOLUTION,
    device=DEVICE,
    num_classes=NUM_CLASSES
)

if hasattr(inference_model, "optimize_for_inference"):
    print("   • Running inference_model.optimize_for_inference()...")
    inference_model.optimize_for_inference()
    print("   ✅ Model optimized for inference speed.")
elif hasattr(inference_model, "model") and hasattr(inference_model.model, "eval"):
    inference_model.model.eval()
    print("   ✅ Model set to eval mode.")

logger.info("Inference model ready and optimized.")


In [ ]:
# CELL 20 — Test Inference with NMS Post-Processing & Supervision
print("=" * 70)
print(f"🎯 [CELL 20] Running test inference (Confidence={CONFIDENCE}, NMS={NMS_THRESHOLD})...")
print("=" * 70)

test_images_dir = DATASET_DIR / "test" / "images"
test_image_paths = sorted(list(test_images_dir.glob("*.*")))
print(f"   • Total test images to evaluate: {len(test_image_paths)}")

predictions_by_image = {}
for idx, p in enumerate(test_image_paths):
    try:
        raw_preds = inference_model.predict(
            source=str(p),
            conf=CONFIDENCE,
            resolution=RESOLUTION
        )
        if hasattr(raw_preds, "with_nms"):
            post_preds = raw_preds.with_nms(threshold=NMS_THRESHOLD)
        else:
            post_preds = raw_preds

        predictions_by_image[p.name] = {
            "boxes_xyxy": post_preds.xyxy.tolist() if hasattr(post_preds, "xyxy") else [],
            "confidence": post_preds.confidence.tolist() if hasattr(post_preds, "confidence") and post_preds.confidence is not None else [],
            "class_id": post_preds.class_id.tolist() if hasattr(post_preds, "class_id") and post_preds.class_id is not None else []
        }
    except Exception as e:
        predictions_by_image[p.name] = {"error": str(e), "boxes_xyxy": [], "confidence": [], "class_id": []}
    
    if (idx + 1) % 50 == 0 or (idx + 1) == len(test_image_paths):
        print(f"   • Evaluated: {idx + 1}/{len(test_image_paths)} images...")

total_preds = sum(len(v["boxes_xyxy"]) for v in predictions_by_image.values())
print(f"✅ Inferred {len(predictions_by_image)} images. Total detections: {total_preds}")
logger.info(f"Test inference complete: {total_preds} detections across {len(test_image_paths)} images.")


In [ ]:
# CELL 21 — Save Prediction JSON
print("=" * 70)
print("💾 [CELL 21] Saving test predictions...")
print("=" * 70)

pred_out_file = INFERENCE_OUTPUT_DIR / f"test_predictions_{MODE_TAG}.json"
with open(pred_out_file, "w", encoding="utf-8") as f:
    json.dump({
        "pipeline": PIPELINE_NAME,
        "mode": MODE_TAG,
        "num_classes": NUM_CLASSES,
        "categories": categories_def,
        "confidence_threshold": CONFIDENCE,
        "nms_threshold": NMS_THRESHOLD,
        "predictions": predictions_by_image
    }, f, indent=2)

print(f"   • Saved predictions to: {pred_out_file}")
logger.info(f"Saved predictions to {pred_out_file}")


In [ ]:
# CELL 22 — Visual Preview of Predictions
print("=" * 70)
print("🖼️  [CELL 22] Rendering test prediction visuals with supervision...")
print("=" * 70)

color_palette = sv.ColorPalette.from_hex([
    "#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"
])
box_annotator = sv.BoxAnnotator(color=color_palette, thickness=2)
label_annotator = sv.LabelAnnotator(color=color_palette, text_scale=0.5, text_thickness=1)

preview_count = 0
for p in test_image_paths[:3]:
    img = cv2.imread(str(p))
    if img is None:
        continue
    preds = predictions_by_image.get(p.name, {})
    xyxy = np.array(preds.get("boxes_xyxy", []), dtype=np.float32)
    conf = np.array(preds.get("confidence", []), dtype=np.float32)
    cids = np.array(preds.get("class_id", []), dtype=int)
    if len(xyxy) > 0:
        det = sv.Detections(xyxy=xyxy, confidence=conf, class_id=cids)
        lbls = [f"{id_to_cat.get(c, str(c))} {cf:.2f}" for c, cf in zip(cids, conf)]
        annotated = box_annotator.annotate(scene=img.copy(), detections=det)
        annotated = label_annotator.annotate(scene=annotated, detections=det, labels=lbls)
    else:
        annotated = img

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"Multi-Class Detection Preview: {p.name} ({len(xyxy)} detected)")
    plt.axis("off")
    plt.show()
    preview_count += 1

print(f"✅ Rendered {preview_count} test preview(s).")


In [ ]:
# CELL 23 — Final Execution Summary
print("=" * 70)
print(f"🏁 [CELL 23] Execution Summary: {PIPELINE_NAME}")
print("=" * 70)
print(f"   • Pipeline Mode:        {MODE_TAG.upper()}")
print(f"   • Num Classes:          {NUM_CLASSES} ({AUTO_CATEGORIES})")
print(f"   • Architecture:         RFDETRBase (resolution={RESOLUTION})")
print(f"   • Optimizer & Schedule: {OPTIMIZER.upper()} (lr={LR}, decay={WEIGHT_DECAY}, cosine decay)")
print(f"   • Early Stopping:       Patience={EARLY_STOPPING_PATIENCE}, Delta={EARLY_STOPPING_MIN_DELTA}")
print(f"   • Dataset Folder:       {DATASET_DIR}")
print(f"   • Output Folder:        {OUTPUT_DIR}")
print(f"   • Exported Model:       {dest_checkpoint}")
print(f"   • Test Predictions:     {pred_out_file}")
print("=" * 70)
print("🎉 Multi-Class Pipeline execution completed successfully!")
logger.info("Multi-class pipeline completed successfully.")
